# S01 – Raw sources loaded and mapped to the common schema

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S01_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

SOURCE = "../../00_source_data/"
OUTPUT = "../S01.xlsx"
COMMENTS = "../S01_comments.xlsx"
AUDIT = "../S01_audit.xlsx"
STEP = "S01"
SCHEMA = ['No', 'Material', 'Type', 'Shape', 'Coat/Functional Group', 'Synthesis_Method', 'Surface_Charge', 'Zeta potential (mV)', 'Diameter (nm)', 'Size_in_Water (nm)', 'Size_in_Medium (nm)', 'Dry_Diameter (nm)', 'DLS_Diameter_Water (nm)', 'DLS_Diameter_Medium (nm)', 'DLS_Diameter_Other (nm)', 'DLS_Diameter (nm)', 'Size_Measurement_Comments', 'Zeta_in_Water (mV)', 'Zeta_in_Medium (mV)', 'Zeta_in_Other (mV)', 'Cell_Type', 'Cancer', 'Cell_organ', 'Cell_type_family', 'Germ_layer', 'No_of_Cells (cells/well)', 'Time (hr)', 'Concentration (ug/ml)', 'Concentration (g/l)2', 'Concentration (uM)', 'Test_raw', 'Serum_Status', 'Test', 'Test_norm', 'Test_family', 'Test_Indicator_raw', 'Test_Indicator', 'Aspect_Ratio', 'Cell_Viability (%)', 'DOI', 'Animal_mapped_combined', 'Animal_mapping_source_combined', 'Dataset']
GUL, LABOUTA, NANOPHAROS = "Gul. Et al..", "Labouta Et. Al. ", "Nanopharos"

In [2]:
# Gul et al.: sheet "Viability"
g = pd.read_excel(SOURCE + "Nano_2021_dataset_raw.xlsx", sheet_name="Viability")
gul = pd.DataFrame({c: [np.nan] * len(g) for c in ["raw_id"] + SCHEMA}).astype(object)
gul["raw_id"] = "G" + g["No"].astype(int).astype(str)
gul["No"] = g["No"].astype(int)
for c in ["Material", "Type", "Shape", "Coat/Functional Group", "Synthesis_Method", "Surface_Charge", "Diameter (nm)", "Size_in_Water (nm)",
          "Size_in_Medium (nm)", "Zeta_in_Water (mV)", "Zeta_in_Medium (mV)", "Cell_Type", "No_of_Cells (cells/well)", "Time (hr)",
          "Concentration (ug/ml)", "Concentration (g/l)2", "Aspect_Ratio", "Cell_Viability (%)", "DOI"]:
    gul[c] = g[c].values
gul["Test_raw"] = g["Test"].values
gul["Test_Indicator_raw"] = g["Test_Indicator"].values
gul["Animal_mapped_combined"] = g["Cell_Source"].astype(str).str.strip().str.lower().values
gul["Animal_mapping_source_combined"] = "Final dataset.xlsx"   # the value of the Gul source column Cell_Source
gul["Dataset"] = GUL
gul["_src"] = GUL

# Labouta et al.: sheet "Master sheet"; raw_id = the Excel row number
l = pd.read_excel(SOURCE + "nn8b07562_si_001(2).xlsx", sheet_name="Master sheet")
lab = pd.DataFrame({c: [np.nan] * len(l) for c in ["raw_id"] + SCHEMA}).astype(object)
lab["raw_id"] = "L" + (l.index + 2).astype(str)
lab["No"] = l["Particle ID"].values
lab["Material"] = l["Nanoparticle"].values
lab["Type"] = l["Type: Organic (O)/inorganic (I)"].values
lab["Coat/Functional Group"] = l["coat"].values
lab["Zeta potential (mV)"] = l["Zeta potential (mV)"].values
lab["Diameter (nm)"] = l["Diameter (nm)"].values
lab["Cell_Type"] = l["Cells"].values
lab["Time (hr)"] = l["Exposure time (h)"].values
lab["Concentration (uM)"] = l["Concentration μM"].values
lab["Test_raw"] = l["Test"].values
lab["Test_Indicator_raw"] = l["Test indicator"].values
lab["Cell_Viability (%)"] = pd.to_numeric(l["% Cell viability"], errors="coerce").values
lab["DOI"] = l["Reference DOI"].values
animal = np.where(l["Human(H)/Animal(A) cells"].astype(str).str.strip().eq("H"), "human", l["Animal?"].astype(str).str.strip().str.lower())
lab["Animal_mapped_combined"] = pd.Series(animal).where(lambda s: s != "nan").values   # a missing species stays missing
lab["Animal_mapping_source_combined"] = "original source columns"
lab["Dataset"] = LABOUTA
lab["_src"] = LABOUTA

# NanoPharos
p = pd.read_excel(SOURCE + "NanoPharos_dataset_raw.xlsx")
nph = pd.DataFrame({c: [np.nan] * len(p) for c in ["raw_id"] + SCHEMA}).astype(object)
nph["raw_id"] = "P" + p["No."].astype(str)
nph["No"] = p["No."].values
nph["Material"] = p["Material"].values
nph["Coat/Functional Group"] = p["Coat"].values
nph["Diameter (nm)"] = p["Diameter (nm)"].values
nph["Zeta_in_Water (mV)"] = p["Zeta potential (mV)"].values   # Zhang et al. 2012 measured the zeta potential in water
nph["Cell_Type"] = p["Cell_type"].values
nph["Cancer"] = p["Cancer_cell"].map({"Cancer": "Yes", "Normal": "No"}).values
nph["Cell_organ"] = p["Cell_organ"].values
nph["Time (hr)"] = p["Time (h)"].astype(str).str.replace("h", "", regex=False).astype(float).values
nph["Concentration (g/l)2"] = p["Concentration (g/L)"].values
nph["Test_raw"] = p["Test"].values
nph["Test_Indicator_raw"] = p["Test_indicator"].values
nph["Cell_Viability (%)"] = p["Viability (%)"].values
nph["Animal_mapped_combined"] = p["Animal"].astype(str).str.lower().values
nph["Animal_mapping_source_combined"] = "Final dataset.xlsx"
nph["Dataset"] = NANOPHAROS
nph["_src"] = NANOPHAROS

In [3]:
df = pd.concat([gul, lab, nph], ignore_index=True)
# keys from the raw rows, used by the rules of the later steps to find records
df["_doi"] = df["DOI"].map(lambda v: "" if pd.isna(v) else str(v).strip().lower())
df["_mat_raw"] = df["Material"].astype(str).str.strip()
df["_coat_raw"] = df["Coat/Functional Group"].map(lambda v: "" if pd.isna(v) else str(v).strip())
df["_diam_raw"] = df["Diameter (nm)"]
df["_cell_raw"] = df["Cell_Type"].astype(str).str.strip()
df["_test_raw"] = df["Test_raw"].astype(str).str.strip()
df["_ind_raw"] = df["Test_Indicator_raw"].map(lambda v: "" if pd.isna(v) else str(v).strip())
df["_first_change"] = np.nan
df = df[["raw_id"] + SCHEMA + ['_src', '_doi', '_mat_raw', '_coat_raw', '_diam_raw', '_cell_raw', '_test_raw', '_ind_raw', '_first_change']]

/tmp/ipykernel_1437321/1630099612.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([gul, lab, nph], ignore_index=True)


In [4]:
audit = Audit(STEP, df)   # S01 changes no value: its comments table is empty and so is its audit workbook
apply_comments(audit, COMMENTS, [])

0 comments applied (): 0 values set, 0 records removed


In [5]:
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

7501 records after the step | 0 records removed | 0 cells changed | ../S01_audit.xlsx
